In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torch.optim as optim
import os
import io
from PIL import Image
import urllib.request
import torchvision.models
import random
import numpy as np
from torch.utils.data import DataLoader, Dataset
import time
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [2]:
#Baseline model for feature extractor

class BaselineFeatureExtractor(nn.Module):
    def __int__(self, input_size, hidden_size, output_size):
        super(BaselineFeatureExtractor, self).__init__()
        
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size//2)
        self.linear3 = nn.Linear(hidden_size//2, output_size)
    
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        x = self.linear3(x)
        return x

In [3]:
#Loading dataset

file_path = "../Datasets/major-crime-indicators.csv"
data = pd.read_csv(file_path)

In [27]:
data['OCC_YEAR'].unique()

array([2014., 2013., 2012.,   nan, 2003., 2011., 2004., 2010., 2009.,
       2008., 2006., 2000., 2005., 2002., 2001., 2015., 2007., 2016.,
       2017., 2018., 2019., 2020., 2021., 2022., 2023., 2024.])

In [4]:
label_encoder = LabelEncoder()

categories = data['OFFENCE'].unique().tolist()
organized_data = []
for category in categories:
    temp_list = data[data['OFFENCE'] == category]
    organized_data.append(temp_list)



shuffled_data = data.sample(frac=1).reset_index(drop=True)

shuffled_data.drop('_id', axis=1, inplace=True)
shuffled_data.drop('EVENT_UNIQUE_ID', axis=1, inplace=True)
shuffled_data.drop('NEIGHBOURHOOD_158', axis=1, inplace=True)
shuffled_data.drop('HOOD_140', axis=1, inplace=True)
shuffled_data.drop('NEIGHBOURHOOD_140', axis=1, inplace=True)
shuffled_data.drop('REPORT_DATE', axis=1, inplace=True)
shuffled_data.drop('OCC_DATE', axis=1, inplace=True)

shuffled_data['DIVISION'] = shuffled_data['DIVISION'].apply(lambda x: int(x[1:]) if x != 'NSA' else 10)
shuffled_data['HOOD_158'] = shuffled_data['HOOD_158'].apply(lambda x: int(x) if x != 'NSA' else 0)

shuffled_data['REPORT_MONTH'] = label_encoder.fit_transform(shuffled_data['REPORT_MONTH'])
shuffled_data['REPORT_DOW'] = label_encoder.fit_transform(shuffled_data['REPORT_DOW'])
shuffled_data['OCC_MONTH'] = label_encoder.fit_transform(shuffled_data['OCC_MONTH'])
shuffled_data['OCC_DOW'] = label_encoder.fit_transform(shuffled_data['OCC_DOW'])
shuffled_data['LOCATION_TYPE'] = label_encoder.fit_transform(shuffled_data['LOCATION_TYPE'])
shuffled_data['PREMISES_TYPE'] = label_encoder.fit_transform(shuffled_data['PREMISES_TYPE'])
shuffled_data['OFFENCE'] = label_encoder.fit_transform(shuffled_data['OFFENCE'])
shuffled_data['MCI_CATEGORY'] = label_encoder.fit_transform(shuffled_data['MCI_CATEGORY'])


In [7]:
input_list = []

for index, row in shuffled_data.iterrows():
    tensor = []
    for value in row.values:
        tensor.append(torch.tensor(value, dtype=torch.float32))
    input_list.append(tensor)

In [11]:
print(len(input_list))

408928


In [12]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.conv1 = nn.Conv1d(1, 2, 1)
        self.conv2 = nn.Conv1d(2, 3, 1)
        self.conv3 = nn.Conv1d(3, 5, 2)
        self.conv4 = nn.Conv1d(5, 7, 2)
        self.conv5 = nn.Conv1d(7, 10, 3)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(F.relu(self.conv4(x)))
        x = self.pool(F.relu(self.conv5(x)))
        
        return x

In [13]:
class Classifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Classifier, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.linear2(x)
        
        return x

In [17]:
extractor = FeatureExtractor()
classifier = Classifier(len(input_list[0]), 100, len(categories))

In [41]:
yearly_list = [[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
for i in range(len(input_list)):
    if (input_list[i][6] in range(2000, 2025)):
        yearly_list[int(input_list[i][6]) - 2000].append(input_list[i])
    else:
        yearly_list[int(input_list[i][0]) - 2000].append(input_list[i])

In [43]:
yearly_list = yearly_list[14:]

In [48]:
test_data = yearly_list[-1]
train_and_valid_data = []
for i in range(10):
    train_and_valid_data += yearly_list[i]
random.shuffle(train_and_valid_data)

In [56]:
valid_data = train_and_valid_data[:82875]
train_data = train_and_valid_data[82875:]

In [67]:
valid_label = []
for i in range(len(valid_data)):
    valid_label.append(valid_data[i][17])
    valid_data[i] = valid_data[i][:17] + valid_data[i][18:]

train_label = []
for i in range(len(train_data)):
    train_label.append(train_data[i][17])
    train_data[i] = train_data[i][:17] + train_data[i][18:]
    
test_label = []
for i in range(len(test_data)):
    test_label.append(test_data[i][17])
    test_data[i] = test_data[i][:17] + test_data[i][18:]

In [68]:
training_data = []
training_label = []
for i in range(len(train_data)//51):
    temp_data = train_data[i*51:i*51 + 51]
    temp_label = train_label[i*51:i*51 + 51]
    training_label.append(temp_label)
    training_data.append(temp_data)

In [69]:
class CombinedCNN(nn.Module):
    def __init__(self, extractor, classifier):
        super(CombinedCNN, self).__init__()
        self.extractor = extractor
        self.classifier = classifier
        
    def forward(self, x):
        x = self.extractor(x)
        x = self.classifier(x)
        
        return x

In [ ]:
def train_model(feature_extractor, classifier, train_loader, train_labels , num_epochs, learning_rate):
    model = CombinedCNN(feature_extractor, classifier)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        for i in range(len(train_loader)):

            outputs = model(train_loader[i])
            loss = criterion(outputs, train_labels[i])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            print('Epoch:', epoch, ', Loss:', loss.item())

    print("Training complete.")
    return model

In [71]:
final_model = train_model(extractor, classifier, training_data, training_label, 30, 0.5)

TypeError: conv1d() received an invalid combination of arguments - got (list, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!list of [list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list, list]!, !Parameter!, !Parameter!, !tuple of (int,)!, !tuple of (int,)!, !tuple of (int,)!, !int!)
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the arguments have invalid types: (!list of